# Train SigExt — Full 27-Model Matrix

Trains **27 SigExt models** across the complete experimental matrix:

| Language | Base Model | Sample Sizes | Thresholds |
|---|---|---|---|
| **English** | `allenai/longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **English** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **Italian** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |

**Seed**: 42 (fixed for all). All models pushed to HuggingFace Hub.

**Optimizations**: datasets loaded once per language, SBERT similarities
computed once, tokenization reused per base model. Checkpointing allows
safe resumption if interrupted.

In [ ]:
import warnings, os, logging
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

try:
    import transformers, datasets
    transformers.utils.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
    datasets.utils.logging.set_verbosity_error()
    datasets.utils.logging.disable_progress_bar()
    logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
except ImportError: pass

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)

In [ ]:
from sm_sip.config import TrainingConfig, SigExtConfig

configs = []
for lang, ld in SigExtConfig.LANG_DATASETS.items():
    for base_key in SigExtConfig.LANG_BASE_MODELS[lang]:
        base_model_id = SigExtConfig.BASE_MODELS[base_key]
        for n in SigExtConfig.SAMPLE_SIZES:
            for t in SigExtConfig.THRESHOLDS:
                name_n = f"{n // 1000}k" if n >= 1000 and n % 1000 == 0 else str(n)
                t_str = f"{t:.2f}".replace(".", "")
                configs.append(TrainingConfig(
                    lang=lang,
                    base_model_id=base_model_id,
                    dataset_name=ld["dataset"],
                    num_samples=n,
                    similarity_threshold=t,
                    output_model_name=f"sigext-{ld[\"prefix\"]}-{lang}-{base_key}-{name_n}-{t_str}t",
                    push_to_hub=True,
                    seed=SigExtConfig.SEED,
                ))

print(f"Total training configs: {len(configs)}")

In [ ]:
from sm_sip.pipelines.training import run_training_matrix

run_training_matrix(configs)